# Aneurysm Volume Prediction v2.5D (Based on v2_attention_plus=0.7976)

这个版本完全沿用你高分版的训练/评估框架，只把主干换成2.5D：
- 分割任务（非回归）
- 分层CV + 多seed + TTA
- Dice+BCE 损失
- OOF阈值+软硬融合+线性校准
- 使用NIfTI spacing计算体积
- 平台路径强适配


In [ ]:
# 如缺包请取消注释
# %pip install nibabel scikit-learn tqdm pandas matplotlib
# %pip install torch torchvision torchaudio


In [ ]:
import os
import sys
import json
import random
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
torch.backends.cudnn.benchmark = True

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = (DEVICE.type == "cuda")
print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("Device:", DEVICE, "AMP:", USE_AMP)


In [ ]:
# ===== 路径自动适配 =====
DATA_ROOT = os.environ.get("ANEURYSM_DATA_ROOT", "").strip()

def is_valid_dataset_dir(p: Path) -> bool:
    return (p / "train").exists() and (p / "test").exists() and (p / "train_labels").exists()

def find_base_dir():
    if DATA_ROOT:
        p = Path(DATA_ROOT)
        if is_valid_dataset_dir(p):
            return p
    fixed = [
        "/dataset/public",
        "/dataset",
        "/kaggle/input/aneurysm-volume-prediction",
        "/kaggle/input/aneurysm-volume",
        "/Users/songling/Desktop/Aneurysm Volume Prediction",
    ]
    for x in fixed:
        p = Path(x)
        if is_valid_dataset_dir(p):
            return p
    for root in [Path("/dataset"), Path("/kaggle/input"), Path.cwd()]:
        if not root.exists():
            continue
        for d in root.rglob("*"):
            if d.is_dir() and is_valid_dataset_dir(d):
                return d
    return None

BASE_DIR = find_base_dir()
if BASE_DIR is None:
    raise FileNotFoundError("未找到数据目录，请设置 ANEURYSM_DATA_ROOT")

TRAIN_IMG_DIR = BASE_DIR / "train"
TRAIN_MASK_DIR = BASE_DIR / "train_labels"
TEST_IMG_DIR = BASE_DIR / "test"
TRAIN_CSV = BASE_DIR / "train.csv"

if Path("/root/setup/solution/working").exists():
    OUTPUT_ROOT = Path("/root/setup/solution/working")
elif Path("/working").exists():
    OUTPUT_ROOT = Path("/working")
elif Path("/kaggle/working").exists():
    OUTPUT_ROOT = Path("/kaggle/working")
else:
    OUTPUT_ROOT = BASE_DIR / "working"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_NAME = os.environ.get("RUN_NAME", "").strip() or datetime.now().strftime("v2_5d_%Y%m%d_%H%M%S")
EXP_DIR = OUTPUT_ROOT / RUN_NAME
CKPT_DIR = EXP_DIR / "checkpoints"
PRED_DIR = EXP_DIR / "predictions"
LOG_DIR = EXP_DIR / "logs"
for d in [EXP_DIR, CKPT_DIR, PRED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("EXP_DIR:", EXP_DIR)


In [ ]:
def load_nii(path):
    nii = nib.load(str(path))
    arr = nii.get_fdata(dtype=np.float32)
    spacing = nii.header.get_zooms()[:3]
    return arr, spacing

def robust_zscore(x, eps=1e-6):
    lo, hi = np.percentile(x, [0.5, 99.5])
    x = np.clip(x, lo, hi)
    m, s = x.mean(), x.std()
    return (x - m) / (s + eps)

def parse_pid(path):
    return int(Path(path).name.split(".")[0])

def volume_from_binary_mask(mask_dhw, spacing):
    voxel = float(spacing[0] * spacing[1] * spacing[2])
    return float(max(mask_dhw.sum() * voxel, 0.0))

def volume_from_prob(prob_dhw, spacing):
    voxel = float(spacing[0] * spacing[1] * spacing[2])
    return float(max(prob_dhw.sum() * voxel, 0.0))

def volumetric_similarity(y_true, y_pred, eps=1e-4):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(1.0 - np.abs(y_true - y_pred) / (y_true + y_pred + eps)))


In [ ]:
class CaseStore:
    def __init__(self):
        self.img = {}
        self.mask = {}
        self.spacing = {}

    def load(self, img_path, mask_path=None):
        pid = parse_pid(img_path)
        if pid not in self.img:
            arr, sp = load_nii(img_path)
            arr = robust_zscore(arr)
            arr = np.transpose(arr, (2,0,1)).astype(np.float32)  # D,H,W
            self.img[pid] = arr
            self.spacing[pid] = np.asarray(sp, dtype=np.float32)
        if mask_path is not None and pid not in self.mask:
            m, _ = load_nii(mask_path)
            m = (m > 0.5).astype(np.float32)
            m = np.transpose(m, (2,0,1)).astype(np.float32)
            self.mask[pid] = m
        return pid

store = CaseStore()

class SliceDataset25D(Dataset):
    def __init__(self, img_paths, mask_paths=None, augment=False):
        self.augment = augment
        self.has_mask = mask_paths is not None
        self.items = []  # (pid,z)
        if self.has_mask:
            for ip, mp in zip(img_paths, mask_paths):
                pid = store.load(ip, mp)
                D = store.img[pid].shape[0]
                self.items.extend([(pid, z) for z in range(D)])
        else:
            for ip in img_paths:
                pid = store.load(ip, None)
                D = store.img[pid].shape[0]
                self.items.extend([(pid, z) for z in range(D)])

    def __len__(self):
        return len(self.items)

    def _build_25d(self, vol, z):
        D = vol.shape[0]
        z0 = max(0, z-1); z1 = z; z2 = min(D-1, z+1)
        return np.stack([vol[z0], vol[z1], vol[z2]], axis=0).astype(np.float32)

    def _aug(self, x, y):
        if random.random() < 0.5:
            x = np.flip(x, axis=1).copy(); y = np.flip(y, axis=1).copy() if y is not None else None
        if random.random() < 0.5:
            x = np.flip(x, axis=2).copy(); y = np.flip(y, axis=2).copy() if y is not None else None
        if random.random() < 0.5:
            k = random.randint(0,3)
            x = np.rot90(x, k=k, axes=(1,2)).copy(); y = np.rot90(y, k=k, axes=(1,2)).copy() if y is not None else None
        if random.random() < 0.7:
            x = x * (1.0 + random.uniform(-0.15, 0.15)) + random.uniform(-0.12, 0.12)
        if random.random() < 0.3:
            x = x + np.random.randn(*x.shape).astype(np.float32) * 0.03
        return x, y

    def __getitem__(self, idx):
        pid, z = self.items[idx]
        vol = store.img[pid]
        x = self._build_25d(vol, z)
        y = None
        if self.has_mask:
            y = store.mask[pid][z][None,...].astype(np.float32)
        if self.augment:
            x, y = self._aug(x, y)
        out = {"image": torch.from_numpy(x), "pid": pid, "z": z}
        if y is not None:
            out["mask"] = torch.from_numpy(y)
        return out


In [ ]:
def gn(ch):
    g=8
    while ch % g != 0 and g > 1:
        g -= 1
    return nn.GroupNorm(g, ch)

class SE2D(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        hid = max(ch // r, 4)
        self.net = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(ch, hid, 1), nn.ReLU(inplace=True), nn.Conv2d(hid, ch, 1), nn.Sigmoid())
    def forward(self, x):
        return x * self.net(x)

class ResBlock2D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.c1 = nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.n1 = gn(out_ch)
        self.c2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.n2 = gn(out_ch)
        self.se = SE2D(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.proj = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        i = self.proj(x)
        x = self.act(self.n1(self.c1(x)))
        x = self.n2(self.c2(x))
        x = self.se(x)
        return self.act(x + i)

class AttnGate2D(nn.Module):
    def __init__(self, x_ch, g_ch, inter_ch):
        super().__init__()
        self.wx = nn.Conv2d(x_ch, inter_ch, 1, bias=False)
        self.wg = nn.Conv2d(g_ch, inter_ch, 1, bias=False)
        self.psi = nn.Sequential(nn.ReLU(inplace=True), nn.Conv2d(inter_ch, 1, 1), nn.Sigmoid())
    def forward(self, x, g):
        return x * self.psi(self.wx(x) + self.wg(g))

class AttentionUNet25D(nn.Module):
    def __init__(self, in_ch=3, base=32):
        super().__init__()
        self.e1 = ResBlock2D(in_ch, base)
        self.p1 = nn.MaxPool2d(2)
        self.e2 = ResBlock2D(base, base*2)
        self.p2 = nn.MaxPool2d(2)
        self.e3 = ResBlock2D(base*2, base*4)
        self.p3 = nn.MaxPool2d(2)
        self.b = ResBlock2D(base*4, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2,2)
        self.a3 = AttnGate2D(base*4, base*4, base*2)
        self.d3 = ResBlock2D(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2,2)
        self.a2 = AttnGate2D(base*2, base*2, base)
        self.d2 = ResBlock2D(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2,2)
        self.a1 = AttnGate2D(base, base, max(base//2, 4))
        self.d1 = ResBlock2D(base*2, base)
        self.head = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1=self.e1(x); e2=self.e2(self.p1(e1)); e3=self.e3(self.p2(e2)); b=self.b(self.p3(e3))
        d3=self.u3(b); d3=self.d3(torch.cat([d3, self.a3(e3,d3)], dim=1))
        d2=self.u2(d3); d2=self.d2(torch.cat([d2, self.a2(e2,d2)], dim=1))
        d1=self.u1(d2); d1=self.d1(torch.cat([d1, self.a1(e1,d1)], dim=1))
        return self.head(d1)

class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight=4.0, bce_weight=0.45, smooth=1e-5):
        super().__init__()
        self.pos_weight = pos_weight
        self.bce_weight = bce_weight
        self.smooth = smooth
    def forward(self, logits, target):
        pw = torch.tensor([self.pos_weight], device=logits.device, dtype=logits.dtype)
        bce = nn.functional.binary_cross_entropy_with_logits(logits, target, pos_weight=pw)
        p = torch.sigmoid(logits).reshape(logits.size(0), -1)
        t = target.reshape(target.size(0), -1)
        inter = (p*t).sum(dim=1)
        den = p.sum(dim=1) + t.sum(dim=1)
        dice = 1.0 - (2.0*inter + self.smooth)/(den + self.smooth)
        return self.bce_weight * bce + (1.0 - self.bce_weight) * dice.mean()


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
train_df["patient_id"] = train_df["patient_id"].astype(int)
all_train_imgs = sorted(TRAIN_IMG_DIR.glob("*.nii.gz"))
all_train_msks = [TRAIN_MASK_DIR / p.name for p in all_train_imgs]
all_test_imgs = sorted(TEST_IMG_DIR.glob("*.nii.gz"))
print("train:", len(all_train_imgs), "test:", len(all_test_imgs))

# preload all train cases to speed up slice iteration
for ip, mp in zip(all_train_imgs, all_train_msks):
    store.load(ip, mp)


In [ ]:
CFG = {
    "n_splits": 5,
    "epochs": 55,
    "batch_size": 32,
    "num_workers": 0,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "base_ch": 32,
    "grad_clip": 1.0,
    "tta": True,
    "early_stop_patience": 8,
    "seeds": [42, 2025],
    "fold_weight_power": 2.0,
}
print(CFG)


In [ ]:
def make_splits(train_df, n_splits=5, seed=42):
    vols = train_df.sort_values("patient_id")["volume"].values
    try:
        bins = pd.qcut(vols, q=5, labels=False, duplicates="drop")
    except Exception:
        bins = pd.cut(vols, bins=5, labels=False)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(np.arange(len(all_train_imgs)), bins))

def infer_slice_tta(model, xb):
    p0 = torch.sigmoid(model(xb))
    p1 = torch.sigmoid(model(torch.flip(xb, dims=[2]))); p1 = torch.flip(p1, dims=[2])
    p2 = torch.sigmoid(model(torch.flip(xb, dims=[3]))); p2 = torch.flip(p2, dims=[3])
    return (p0 + p1 + p2) / 3.0

def predict_case_prob(model, vol_dhw, batch=64, tta=True):
    D,H,W = vol_dhw.shape
    xs=[]
    for z in range(D):
        z0=max(0,z-1); z1=z; z2=min(D-1,z+1)
        xs.append(np.stack([vol_dhw[z0], vol_dhw[z1], vol_dhw[z2]], axis=0).astype(np.float32))
    xs = np.stack(xs, axis=0)
    probs=[]
    model.eval()
    with torch.no_grad():
        for i in range(0, D, batch):
            xb = torch.from_numpy(xs[i:i+batch]).to(DEVICE)
            pb = infer_slice_tta(model, xb) if tta else torch.sigmoid(model(xb))
            probs.append(pb.cpu().numpy())
    probs = np.concatenate(probs, axis=0)[:,0]
    return probs.astype(np.float32)

def depth_aggregate(prob_dhw):
    D = prob_dhw.shape[0]
    out = np.zeros_like(prob_dhw, dtype=np.float32)
    for z in range(D):
        z0=max(0,z-1); z1=z; z2=min(D-1,z+1)
        out[z] = 0.25*prob_dhw[z0] + 0.5*prob_dhw[z1] + 0.25*prob_dhw[z2]
    return out

def train_one_fold(seed, fold, tr_idx, va_idx):
    seed_everything(seed + fold)
    tr_ds = SliceDataset25D([all_train_imgs[i] for i in tr_idx], [all_train_msks[i] for i in tr_idx], augment=True)
    tr_ld = DataLoader(tr_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"])

    va_pids = [parse_pid(all_train_imgs[i]) for i in va_idx]
    model = AttentionUNet25D(in_ch=3, base=CFG["base_ch"]).to(DEVICE)
    crit = DiceBCELoss(pos_weight=4.0, bce_weight=0.45)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"], eta_min=1e-5)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    sd = CKPT_DIR / f"seed_{seed}"
    sd.mkdir(parents=True, exist_ok=True)
    ckpt = sd / f"fold_{fold}.pt"
    best_vs, no_imp = -1.0, 0

    for ep in range(CFG["epochs"]):
        model.train(); losses=[]
        for b in tr_ld:
            x = b["image"].to(DEVICE)
            y = b["mask"].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logit = model(x)
                loss = crit(logit, y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(opt); scaler.update()
            losses.append(loss.item())
        sch.step()

        # case-level val VS
        model.eval(); gts=[]; prs=[]
        with torch.no_grad():
            for pid in va_pids:
                vol = store.img[pid]
                gt = store.mask[pid]
                sp = store.spacing[pid]
                prob = predict_case_prob(model, vol, batch=64, tta=False)
                prob = depth_aggregate(prob)
                pm = (prob > 0.5).astype(np.uint8)
                gts.append(volume_from_binary_mask(gt, sp))
                prs.append(volume_from_binary_mask(pm, sp))
        vs = volumetric_similarity(gts, prs)
        print(f"seed {seed} fold {fold} ep {ep+1:02d}/{CFG["epochs"]} loss={np.mean(losses):.4f} valVS={vs:.4f}")
        if vs > best_vs:
            best_vs = vs; no_imp = 0; torch.save(model.state_dict(), ckpt)
        else:
            no_imp += 1
        if no_imp >= CFG["early_stop_patience"]:
            print(f"early stop seed {seed} fold {fold}, best={best_vs:.4f}")
            break
    return ckpt, best_vs

def build_oof_items(model_paths_by_seed, splits, model_weights_by_seed_fold):
    items=[]
    for fold, (_, va_idx) in enumerate(splits):
        fold_models=[]; fold_ws=[]
        for seed, plist in model_paths_by_seed.items():
            m = AttentionUNet25D(in_ch=3, base=CFG["base_ch"]).to(DEVICE)
            m.load_state_dict(torch.load(plist[fold], map_location=DEVICE))
            m.eval()
            fold_models.append(m)
            w = float(model_weights_by_seed_fold[(seed, fold)]) ** float(CFG.get("fold_weight_power", 1.0))
            fold_ws.append(w)
        fold_ws = np.asarray(fold_ws, dtype=np.float64)
        fold_ws = fold_ws / (fold_ws.sum() + 1e-12)

        for i in va_idx:
            pid = parse_pid(all_train_imgs[i])
            vol = store.img[pid]
            probs=[]
            with torch.no_grad():
                for m in fold_models:
                    probs.append(predict_case_prob(m, vol, batch=64, tta=CFG["tta"]))
            prob = np.tensordot(fold_ws, np.stack(probs, axis=0), axes=(0,0)).astype(np.float32)
            prob = depth_aggregate(prob)
            items.append({"pid": pid, "spacing": store.spacing[pid], "prob": prob})
    return items


In [ ]:
start = time.time()
splits = make_splits(train_df, n_splits=CFG["n_splits"], seed=42)
model_paths_by_seed = {}
model_weights_by_seed_fold = {}
score_rows = []

for seed in CFG["seeds"]:
    fold_paths = []
    for fold, (tr_idx, va_idx) in enumerate(splits):
        p, s = train_one_fold(seed, fold, tr_idx, va_idx)
        fold_paths.append(p)
        score_rows.append({"seed": seed, "fold": fold, "best_vs": float(s), "ckpt": str(p)})
        model_weights_by_seed_fold[(seed, fold)] = max(float(s), 1e-6)
    model_paths_by_seed[seed] = fold_paths

score_df = pd.DataFrame(score_rows)
score_df.to_csv(LOG_DIR / "fold_scores.csv", index=False)
print(score_df)
print(f"Training time: {(time.time()-start)/60:.1f} min")


In [ ]:
# ===== OOF调参：阈值 + 软硬融合 + 线性校准 =====
oof_items = build_oof_items(model_paths_by_seed, splits, model_weights_by_seed_fold)
gt_map = {int(r.patient_id): float(r.volume) for _, r in train_df.iterrows()}

best = {"vs": -1.0, "thr": 0.5, "alpha": 1.0}
coarse_thr = np.arange(0.28, 0.71, 0.02)
coarse_alpha = np.arange(0.0, 1.01, 0.1)
for thr in coarse_thr:
    hard, soft, y = [], [], []
    for it in oof_items:
        pid = int(it["pid"])
        sp = it["spacing"]
        prob = it["prob"].astype(np.float32)
        hard.append(volume_from_binary_mask((prob > thr).astype(np.uint8), sp))
        soft.append(volume_from_prob(prob, sp))
        y.append(gt_map[pid])
    hard = np.asarray(hard, dtype=np.float64)
    soft = np.asarray(soft, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    for a in coarse_alpha:
        pred = a * hard + (1-a) * soft
        vs = volumetric_similarity(y, pred)
        if vs > best["vs"]:
            best = {"vs": float(vs), "thr": float(thr), "alpha": float(a)}

fine_thr = np.arange(max(0.05, best["thr"]-0.04), min(0.95, best["thr"]+0.04)+1e-9, 0.005)
fine_alpha = np.arange(max(0.0, best["alpha"]-0.2), min(1.0, best["alpha"]+0.2)+1e-9, 0.05)
for thr in fine_thr:
    hard, soft, y = [], [], []
    for it in oof_items:
        pid = int(it["pid"])
        sp = it["spacing"]
        prob = it["prob"].astype(np.float32)
        hard.append(volume_from_binary_mask((prob > thr).astype(np.uint8), sp))
        soft.append(volume_from_prob(prob, sp))
        y.append(gt_map[pid])
    hard = np.asarray(hard, dtype=np.float64)
    soft = np.asarray(soft, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    for a in fine_alpha:
        pred = a * hard + (1-a) * soft
        vs = volumetric_similarity(y, pred)
        if vs > best["vs"]:
            best = {"vs": float(vs), "thr": float(thr), "alpha": float(a)}

print("Best OOF before calibration:", best)

oof_rows=[]
for it in oof_items:
    pid = int(it["pid"])
    sp = it["spacing"]
    prob = it["prob"].astype(np.float32)
    vh = volume_from_binary_mask((prob > best["thr"]).astype(np.uint8), sp)
    vs = volume_from_prob(prob, sp)
    blend = best["alpha"] * vh + (1-best["alpha"]) * vs
    oof_rows.append({"patient_id": pid, "gt": gt_map[pid], "blend": blend})
oof_df = pd.DataFrame(oof_rows).sort_values("patient_id").reset_index(drop=True)
k, b = np.polyfit(oof_df["blend"].values, oof_df["gt"].values, deg=1)
oof_df["calibrated"] = np.clip(k * oof_df["blend"].values + b, 0, None)
vs_uncal = volumetric_similarity(oof_df["gt"].values, oof_df["blend"].values)
vs_cal = volumetric_similarity(oof_df["gt"].values, oof_df["calibrated"].values)
use_cal = bool(vs_cal >= vs_uncal)
print(f"OOF VS uncal={vs_uncal:.6f} | cal={vs_cal:.6f} | use_cal={use_cal}")
oof_df.to_csv(PRED_DIR / "oof_predictions.csv", index=False)


In [ ]:
# ===== Test推理 + 提交输出 =====
models=[]
model_ws=[]
for seed, plist in model_paths_by_seed.items():
    for fold, p in enumerate(plist):
        m = AttentionUNet25D(in_ch=3, base=CFG["base_ch"]).to(DEVICE)
        m.load_state_dict(torch.load(p, map_location=DEVICE))
        m.eval()
        models.append(m)
        w = float(model_weights_by_seed_fold[(seed, fold)]) ** float(CFG.get("fold_weight_power", 1.0))
        model_ws.append(w)
model_ws = np.asarray(model_ws, dtype=np.float64)
model_ws = model_ws / (model_ws.sum() + 1e-12)

for ip in all_test_imgs:
    store.load(ip, None)

rows=[]
for ip in tqdm(all_test_imgs, desc="Inference"):
    pid = parse_pid(ip)
    vol = store.img[pid]
    sp = store.spacing[pid]
    probs=[]
    with torch.no_grad():
        for m in models:
            probs.append(predict_case_prob(m, vol, batch=64, tta=CFG["tta"]))
    prob = np.tensordot(model_ws, np.stack(probs, axis=0), axes=(0,0)).astype(np.float32)
    prob = depth_aggregate(prob)
    vh = volume_from_binary_mask((prob > best["thr"]).astype(np.uint8), sp)
    vs = volume_from_prob(prob, sp)
    blend = best["alpha"] * vh + (1-best["alpha"]) * vs
    pred = np.clip(k * blend + b, 0, None) if use_cal else max(blend, 0.0)
    rows.append({"patient_id": pid, "volume": float(pred)})

sub = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
sub.to_csv(PRED_DIR / "submission.csv", index=False)

targets = [
    Path("/root/setup/solution/working/submission.csv"),
    Path("/working/submission.csv"),
    OUTPUT_ROOT / "submission.csv",
]
saved=[]
for t in targets:
    try:
        t.parent.mkdir(parents=True, exist_ok=True)
        sub.to_csv(t, index=False)
        saved.append(str(t))
    except Exception as e:
        print("skip", t, e)

meta = {
    "run_name": RUN_NAME,
    "cfg": CFG,
    "best_oof": best,
    "calibration": {"use": use_cal, "k": float(k), "b": float(b), "vs_uncal": float(vs_uncal), "vs_cal": float(vs_cal)},
    "saved_submission_paths": saved,
}
with open(LOG_DIR / "run_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Saved submission paths:")
for p in saved:
    print(" -", p)
print(sub.head())


In [ ]:
# 提交前自检
for p in [Path("/root/setup/solution/working/submission.csv"), Path("/working/submission.csv"), PRED_DIR / "submission.csv"]:
    print(p, "exists=", p.exists())
    if p.exists():
        d = pd.read_csv(p)
        print(" shape=", d.shape, " cols=", d.columns.tolist())
        print(d.head(3))


## 对照说明
- 与 `v2_attention_plus` 相同：分层CV、多seed、TTA、OOF阈值+融合+校准、平台路径。
- 不同点：主干改为2.5D Attention U-Net（输入相邻3切片），以减少小样本3D过拟合风险。
